In [2]:
import os
import pandas as pd
from pathlib import Path
import numpy as np
import anndata
import time
import matplotlib.pyplot as plt
import json

from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

pd.set_option('display.max_columns', 500)

In [3]:
version = '20261130'
download_base = Path('../../data/allen-brain-cell-atlas-staging')
abc_cache = AbcProjectCache.from_cache_dir(
    download_base,
    s3_bucket='allen-brain-cell-atlas-staging',
    auth_required=True,
)

abc_cache.load_manifest(f'releases/{version}/manifest.json')

/Users/chris.morrison/src/abc_atlas_access/src/abc_atlas_access/abc_atlas_cache/cloud_cache.py:679: OutdatedManifestWarning: You are loading releases/20260930/manifest.json. A more up to date version of the dataset -- releases/20261130/manifest.json -- exists online. To see the changes between the two versions of the dataset, run
type.compare_manifests('releases/20260930/manifest.json', 'releases/20261130/manifest.json')
To load another version of the dataset, run
type.load_manifest('releases/20261130/manifest.json')
  warnings.warn(msg, OutdatedManifestWarning)


Read in the two DataFrames from the aging dataset we'll need to create an equivalent cluster annotation terms and term set like the WMB and WHB taxonomies.

In [4]:
taxonomy_dir = 'HMBA-Subcortex-taxonomy-Marmoset'

In [5]:
abc_cache.list_metadata_files(taxonomy_dir)

['abbreviation_term',
 'cell_2d_embedding_coordinates',
 'cell_to_cluster_membership',
 'cluster',
 'cluster_annotation_term',
 'cluster_annotation_term_set',
 'cluster_annotation_to_abbreviation_map',
 'cluster_to_cluster_annotation_membership',
 'cluster_to_dominant_region',
 'cluster_to_neurotransmitter',
 'value_sets']

In [6]:
term = abc_cache.get_metadata_dataframe(
    taxonomy_dir,
    'cluster_annotation_term'
)
term

cluster_annotation_term.csv: 100%|██████████| 252k/252k [00:00<00:00, 1.14MMB/s] 


,label,name,cluster_annotation_term_set_label,cluster_annotation_term_set_name,color_hex_triplet,term_order,term_set_order,parent_term_label,parent_term_name,parent_term_set_label,external_label
0,CS20260831_LVL0_00,CNU-HTH Glut,CCN20260831_LVL0,Class,#56B4E9,1,0,NaN,NaN,NaN,NaN
1,CS20260831_LVL0_01,CNU-HTH GABA,CCN20260831_LVL0,Class,#E69F00,2,0,NaN,NaN,NaN,NaN
2,CS20260831_LVL0_02,HTH Glut,CCN20260831_LVL0,Class,#0072B2,3,0,NaN,NaN,NaN,NaN
3,CS20260831_LVL0_03,HTH GABA,CCN20260831_LVL0,Class,#F0E442,4,0,NaN,NaN,NaN,NaN
4,CS20260831_LVL0_04,HTH Hist,CCN20260831_LVL0,Class,#D55E00,5,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1961,CS20260831_LVL2_1732,cluster_830,CCN20260831_LVL2,Cluster,#f2b12f,1733,2,CS20260831_LVL1_211,VLMC,CCN20260831_LVL1,NaN
1962,CS20260831_LVL2_1733,cluster_841,CCN20260831_LVL2,Cluster,#4d12b0,1734,2,CS20260831_LVL1_211,VLMC,CCN20260831_LVL1,NaN
1963,CS20260831_LVL2_1734,cluster_842,CCN20260831_LVL2,Cluster,#aedba2,1735,2,CS20260831_LVL1_211,VLMC,CCN20260831_LVL1,NaN
1964,CS20260831_LVL2_1735,cluster_843,CCN20260831_LVL2,Cluster,#fc1d81,1736,2,CS20260831_LVL1_211,VLMC,CCN20260831_LVL1,NaN


In [7]:
term_sets = abc_cache.get_metadata_dataframe(
    directory=taxonomy_dir,
    file_name='cluster_annotation_term_set'
).set_index('label')
term_sets

cluster_annotation_term_set.csv: 100%|██████████| 183/183 [00:00<00:00, 2.62kMB/s]


,name,description,order,parent_term_set_label
label,,,,
CCN20260831_LVL0,Class,Class,0,NaN
CCN20260831_LVL1,Group,Group,1,CCN20260831_LVL0
CCN20260831_LVL2,Cluster,Cluster,2,CCN20260831_LVL1


In [8]:
filtered = term[pd.notna(term['parent_term_label'])]
first_child = filtered.groupby('parent_term_label')[['label','name','term_order','cluster_annotation_term_set_name']].first()
first_child

,label,name,term_order,cluster_annotation_term_set_name
parent_term_label,,,,
CS20260831_LVL0_00,CS20260831_LVL1_000,BST EBF2 Glut,1,Group
CS20260831_LVL0_01,CS20260831_LVL1_011,ACEN SST GABA,12,Group
CS20260831_LVL0_02,CS20260831_LVL1_023,AHN OTP-FEZF1-ITGA8 Glut,24,Group
CS20260831_LVL0_03,CS20260831_LVL1_044,ArH TBX3 GABA,45,Group
CS20260831_LVL0_04,CS20260831_LVL1_057,TM HDC Hist-GABA,58,Group
...,...,...,...,...
CS20260831_LVL1_207,CS20260831_LVL2_1717,cluster_16,1718,Cluster
CS20260831_LVL1_208,CS20260831_LVL2_1723,cluster_826,1724,Cluster
CS20260831_LVL1_209,CS20260831_LVL2_1726,cluster_837,1727,Cluster


In [9]:
term.set_index('label',inplace=True)
term.loc[first_child.index,'first_child_label'] = first_child['label']
term.loc[first_child.index,'first_child_term_set_name'] = first_child['cluster_annotation_term_set_name']
term.reset_index(inplace=True)

In [10]:
term[pd.notna(term['first_child_label'])]

,label,name,cluster_annotation_term_set_label,cluster_annotation_term_set_name,color_hex_triplet,term_order,term_set_order,parent_term_label,parent_term_name,parent_term_set_label,external_label,first_child_label,first_child_term_set_name
0,CS20260831_LVL0_00,CNU-HTH Glut,CCN20260831_LVL0,Class,#56B4E9,1,0,NaN,NaN,NaN,NaN,CS20260831_LVL1_000,Group
1,CS20260831_LVL0_01,CNU-HTH GABA,CCN20260831_LVL0,Class,#E69F00,2,0,NaN,NaN,NaN,NaN,CS20260831_LVL1_011,Group
2,CS20260831_LVL0_02,HTH Glut,CCN20260831_LVL0,Class,#0072B2,3,0,NaN,NaN,NaN,NaN,CS20260831_LVL1_023,Group
3,CS20260831_LVL0_03,HTH GABA,CCN20260831_LVL0,Class,#F0E442,4,0,NaN,NaN,NaN,NaN,CS20260831_LVL1_044,Group
4,CS20260831_LVL0_04,HTH Hist,CCN20260831_LVL0,Class,#D55E00,5,0,NaN,NaN,NaN,NaN,CS20260831_LVL1_057,Group
...,...,...,...,...,...,...,...,...,...,...,...,...,...
224,CS20260831_LVL1_207,COP,CCN20260831_LVL1,Group,#CCA43D,208,1,CS20260831_LVL0_16,Nonneuron,CCN20260831_LVL0,CS20250428_GROUP_0009,CS20260831_LVL2_1717,Cluster
225,CS20260831_LVL1_208,Endo,CCN20260831_LVL1,Group,#66493D,209,1,CS20260831_LVL0_16,Nonneuron,CCN20260831_LVL0,CS20250428_GROUP_0011,CS20260831_LVL2_1723,Cluster
226,CS20260831_LVL1_209,Pericyte,CCN20260831_LVL1,Group,#B97300,210,1,CS20260831_LVL0_16,Nonneuron,CCN20260831_LVL0,CS20250428_GROUP_0027,CS20260831_LVL2_1726,Cluster
227,CS20260831_LVL1_210,SMC,CCN20260831_LVL1,Group,#858881,211,1,CS20260831_LVL0_16,Nonneuron,CCN20260831_LVL0,CS20250428_GROUP_0028,CS20260831_LVL2_1727,Cluster


In [12]:
membership = abc_cache.get_metadata_dataframe(directory=taxonomy_dir, file_name='cluster_to_cluster_annotation_membership')
pivot = membership.groupby(['cluster_alias', 'cluster_annotation_term_set_name'])['cluster_annotation_term_name'].first().unstack()
pivot = pivot[term_sets['name']] # order columns
pivot.fillna('Other', inplace=True)
pivot.sort_values(['Class', 'Group', 'Cluster'], inplace=True)
cols = pivot.columns.to_list()
pivot.columns = cols
pivot

,Class,Group,Cluster
cluster_alias,,,
cluster_249,BG,AMY-STR ITC-NUDAP eSPN,cluster_249
cluster_254,BG,AMY-STR ITC-NUDAP eSPN,cluster_254
cluster_255,BG,AMY-STR ITC-NUDAP eSPN,cluster_255
cluster_256,BG,AMY-STR ITC-NUDAP eSPN,cluster_256
cluster_257,BG,AMY-STR ITC-NUDAP eSPN,cluster_257
...,...,...,...
cluster_212,THM Glut,VP-MG Glut,cluster_212
cluster_217,THM Glut,VP-MG Glut,cluster_217
cluster_223,THM Glut,VP-MG Glut,cluster_223


In [13]:
lookup = {}
for tag in term_sets['name']:
    #print(tag)
    pred = (term['cluster_annotation_term_set_name'] == tag)
    filtered = term[pred].copy()
    filtered.set_index('name', inplace=True)
    lookup[tag] = filtered

Helper functions to lookup an term attribut and format a cell in the html table

In [14]:
def get_value(c, n, v) :
    return lookup[c].loc[n][v]

def format_cell (df,c,add_id=False,add_plus=False,add_minus=False) :

    divs = pd.DataFrame(index=df.index)
    
    pattern = '<div class="circle" style="background-color:%s"></div>'
    divs['circle'] = [pattern % get_value(c,x,'color_hex_triplet') for x in df[c]]
    
    pattern = '<div class="celltext">%s</div>'
    divs['name'] = [pattern % x for x in df[c]]
   
    divs['id'] = ''
    if add_id :
        pattern = '<div id="%s"></div>'
        divs['id'] = [pattern % get_value(c,x,'label') for x in df[c]]
        
    divs['plus'] = ''
    if add_plus :
        pattern = '<div class="celltext"><a href="%s.html#%s">[+]</a></div>'
        divs['plus'] = [pattern % (get_value(c,x,'first_child_term_set_name'),
                                   get_value(c,x,'first_child_label')) for x in df[c]]
        
    divs['minus'] = ''
    if add_minus :
        pattern = '<div class="celltext"><a href="%s.html#%s">[-]</a></div>'
        divs['minus'] = [pattern % (get_value(c,x,'cluster_annotation_term_set_name'),
                                    get_value(c,x,'label')) for x in df[c]]
    
    cols = ['id','circle','name','plus','minus']
    output = divs[cols].apply(lambda row: ''.join(row.values.astype(str)), axis=1)
    return output


Helper function to create html document

In [15]:
def create_html(df, ts, file, title):
    
    # apply formatter to each term set
    df_formatted = df.copy()
    
    for tag in term_sets['name'] :
        if tag in df_formatted.columns :

            add_id = False
            if tag == ts :
                add_id = True
                
            add_plus = False
            if tag == ts and tag not in ['Cluster']:
                add_plus = True
                
            add_minus = False
            if tag != ts and tag not in ['']:
                add_minus = True
                
            df_formatted[tag] = format_cell(df,tag,add_id,add_plus,add_minus)
            
            
    output = df_formatted.to_html(index=False, na_rep='',
                        render_links=True,escape=False,
                        classes="mystyle")

    html_string = '''
    <html>
    <head><title>%s</title></head>
    <link rel="stylesheet" type="text/css" href="../../simple_style.css"/>
    <body>
    {table}
    </body>
    </html>.
    ''' % title

    # OUTPUT AN HTML FILE
    with open(file, 'w') as f:
        f.write(html_string.format(table=output))

In [16]:
# Write the data to the _static directory of the abc_atlas_access so that links work properly in the jupyter-book/sphinx page.
output_directory = os.path.join('../../_static', taxonomy_dir, version)
os.makedirs(output_directory, exist_ok=True)

In [17]:
df_supertype = pivot[['Class']].copy()
df_supertype.drop_duplicates(inplace=True)

file = os.path.join(output_directory, 'class.html')
title = 'HMBA Subcortex Marmoset taxonomy: Class'
create_html(df_supertype, 'Class', file, title)
print(len(df_supertype))

17


In [19]:
df_supertype = pivot[['Class', 'Group']].copy()
df_supertype.drop_duplicates(inplace=True)

file = os.path.join(output_directory, 'group.html')
title = 'HMBA Subcortex Marmoset taxonomy: Group'
create_html(df_supertype, 'Group', file, title)
print(len(df_supertype))

212


In [20]:
df_supertype = pivot[['Class', 'Group', 'Cluster']].copy()
df_supertype.drop_duplicates(inplace=True)

file = os.path.join(output_directory, 'cluster.html')
title = 'HMBA Subcortex Marmoset taxonomy: Cluster'
create_html(df_supertype, 'Cluster', file, title)
print(len(df_supertype))

1737
